# 03 — GGUF Conversion and 4-Model Evaluation

Converts SFT and DPO models to GGUF Q4_K_M, then benchmarks four models on the held-out test split.

**Prerequisites:** Notebooks 01 and 02 must be complete. `test_split.json` must exist (created by Notebook 00).

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'
DEEPINFRA_API_KEY = ''   # required for teacher model evaluation

SFT_MODEL_DIR  = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
DPO_MODEL_DIR  = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/merged/'
GGUF_DIR       = f'{DRIVE_PATH}gguf/'
EVAL_DIR       = f'{DRIVE_PATH}evaluation/'
TEST_PATH      = f'{DRIVE_PATH}data/test_split.json'
BASELINE_MODEL = f'Qwen/Qwen2.5-{MODEL_SIZE.upper()}-Instruct'
TEACHER_MODEL  = 'deepseek-ai/DeepSeek-R1-Distill-Llama-70B'
# ──────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers accelerate sentence-transformers scikit-learn openai tqdm

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(GGUF_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

In [ ]:
# ── GGUF conversion via llama.cpp (skips if files already exist)
from pathlib import Path

SFT_GGUF = f'{GGUF_DIR}smartriz-sft-{MODEL_SIZE}-Q4_K_M.gguf'
DPO_GGUF = f'{GGUF_DIR}smartriz-dpo-{MODEL_SIZE}-Q4_K_M.gguf'

if not Path('/content/llama.cpp').exists():
    print('Cloning llama.cpp...')
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !pip install -q /content/llama.cpp

if not Path(SFT_GGUF).exists():
    print('Converting SFT model to GGUF Q4_K_M...')
    !python /content/llama.cpp/convert_hf_to_gguf.py \
        {SFT_MODEL_DIR} --outfile {SFT_GGUF} --outtype q4_k_m
print(f'SFT GGUF: {Path(SFT_GGUF).stat().st_size / 1e9:.2f} GB  →  {SFT_GGUF}')

if not Path(DPO_GGUF).exists():
    print('Converting DPO model to GGUF Q4_K_M...')
    !python /content/llama.cpp/convert_hf_to_gguf.py \
        {DPO_MODEL_DIR} --outfile {DPO_GGUF} --outtype q4_k_m
print(f'DPO GGUF: {Path(DPO_GGUF).stat().st_size / 1e9:.2f} GB  →  {DPO_GGUF}')

In [ ]:
import json

with open(TEST_PATH) as f:
    test_cases = json.load(f)['cases']
print(f'Test examples: {len(test_cases)}')

In [ ]:
# ── Evaluation metric helpers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def principle_accuracy(generated, reference_principles):
    """True if model picked at least 2 correct TRIZ principle numbers."""
    ref_nums = set(re.findall(r'#(\d+)', ' '.join(reference_principles)))
    gen_nums = set(re.findall(r'#(\d+)', generated))
    return len(ref_nums & gen_nums) >= 2

def contradiction_accuracy(generated, ref_cp):
    """True if both improving and worsening parameter keywords appear in output."""
    imp_words = [w for w in ref_cp.get('improving_parameter', '').lower().split() if len(w) > 3]
    wor_words = [w for w in ref_cp.get('worsening_parameter', '').lower().split() if len(w) > 3]
    gen_lower = generated.lower()
    imp_ok = any(w in gen_lower for w in imp_words)
    wor_ok = any(w in gen_lower for w in wor_words)
    return imp_ok and wor_ok

def solution_cosine(generated, reference_solution):
    embs = embed_model.encode([generated, reference_solution])
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])

def score_predictions(generations, cases):
    scores = []
    for gen, case in zip(generations, cases):
        scores.append({
            'principle_acc':     principle_accuracy(gen, case['inventive_principles']),
            'contradiction_acc': contradiction_accuracy(gen, case['contradiction_pair']),
            'cosine_sim':        solution_cosine(gen, case['solution']),
            'generated_preview': gen[:300],
        })
    return {
        'principle_acc':     float(np.mean([s['principle_acc'] for s in scores])),
        'contradiction_acc': float(np.mean([s['contradiction_acc'] for s in scores])),
        'cosine_sim':        float(np.mean([s['cosine_sim'] for s in scores])),
        'per_example':       scores,
    }

print('Metric helpers ready')

In [ ]:
# ── Model runner utilities
from transformers import pipeline as hf_pipeline, AutoTokenizer
from openai import OpenAI
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def run_hf_model(model_path, cases, label=''):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    gen = hf_pipeline('text-generation', model=model_path, tokenizer=tok,
                      device_map='auto', max_new_tokens=1024, do_sample=False)
    results = []
    for case in tqdm(cases, desc=f'eval {label or model_path.split("/")[-1]}'):
        prompt = tok.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user',   'content': case['problem']}],
            tokenize=False, add_generation_prompt=True
        )
        out = gen(prompt)[0]['generated_text']
        results.append(out[len(prompt):].strip())
    return results

def run_api_model(model_name, cases, label=''):
    client = OpenAI(api_key=DEEPINFRA_API_KEY,
                    base_url='https://api.deepinfra.com/v1/openai')
    results = []
    for case in tqdm(cases, desc=f'eval {label or model_name}'):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': case['problem']},
                ],
                temperature=0, max_tokens=1024,
            )
            results.append(resp.choices[0].message.content)
        except Exception as e:
            results.append(f'API_ERROR: {e}')
    return results

print('Model runners ready')

In [ ]:
# ── Run evaluation — skips models already in results.json
from pathlib import Path

results_path = f'{EVAL_DIR}results.json'

if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)
    print(f'Loaded existing results for: {list(all_results.keys())}')
else:
    all_results = {}

models_to_eval = [
    ('baseline', 'hf',  BASELINE_MODEL, 'baseline-qwen2.5'),
    ('sft',      'hf',  SFT_MODEL_DIR,  'sft-7b'),
    ('dpo',      'hf',  DPO_MODEL_DIR,  'dpo-7b'),
    ('teacher',  'api', TEACHER_MODEL,  'deepseek-r1-70b'),
]

for name, mode, path, label in models_to_eval:
    if name in all_results:
        print(f'Skipping {name} — already in results.json')
        continue
    print(f'\nEvaluating: {name} ({label})')
    generations = (run_hf_model(path, test_cases, label)
                   if mode == 'hf'
                   else run_api_model(path, test_cases, label))

    all_results[name] = score_predictions(generations, test_cases)

    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)

    r = all_results[name]
    print(f'  Principle Acc:     {r["principle_acc"]:.3f}')
    print(f'  Contradiction Acc: {r["contradiction_acc"]:.3f}')
    print(f'  Cosine Sim:        {r["cosine_sim"]:.3f}')

print(f'\nAll results saved to {results_path}')

In [ ]:
# ── Final comparison table
header = f'{"Model":<12} {"Principle Acc":>15} {"Contradiction Acc":>18} {"Cosine Sim":>12}'
print(header)
print('-' * len(header))

display_order = ['baseline', 'sft', 'dpo', 'teacher']
for name in display_order:
    if name not in all_results:
        continue
    r = all_results[name]
    print(f'{name:<12} {r["principle_acc"]:>15.3f} {r["contradiction_acc"]:>18.3f} {r["cosine_sim"]:>12.3f}')

print(f'\nFull results with per-example breakdowns → {results_path}')